In [ ]:
# ==============================================================================
# BANCADA DE PROVOCAÇÃO: TESTE DE CAUSALIDADE DE GRANGER & GRÁFICOS COMPARATIVOS
# Comparação do EPU (Baker, Bloom & Davis) vs. VIX Tupiniquim (PCA e LASSO)
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

print("--- [BANCADA DE PROVOCAÇÃO]: ANÁLISE COMPLETA (VIX PCA & LASSO vs EPU) ---")

# ==============================================================================
# 1. CARREGAMENTO E ALINHAMENTO DAS SÉRIES HISTÓRICAS
# ==============================================================================
# Carrega a série do EPU do arquivo do Baker, Bloom & Davis
df_epu = pd.read_excel('epu_baker_bloom_davis_brazil.xlsx', sheet_name='Brazil EPU Index')
df_epu['Data'] = pd.to_datetime(df_epu['year'].astype(str) + '-' + df_epu['month'].astype(str) + '-01')
df_epu = df_epu.rename(columns={'Brazil News-Based EPU': 'EPU'})

# Carrega os dois VIX (PCA e LASSO) gerados no pipeline principal
df_vix = pd.read_excel('tabela_vix_tupiniquim_pca.xlsx') 
df_vix['Data'] = pd.to_datetime(df_vix['Data'])

# Purificação Estrutural (STL) do EPU
print("Aplicando Filtro Estrutural STL (period=13) na série do EPU...")
stl_epu = STL(df_epu['EPU'], period=13, robust=True).fit()
df_epu['EPU_SA'] = stl_epu.trend + stl_epu.resid

# Merge e alinhamento amostral
df_analise = pd.merge(df_vix, df_epu[['Data', 'EPU_SA']], on='Data', how='inner').sort_values('Data').reset_index(drop=True)
print(f"[ALINHAMENTO]: Amostra comum final com {len(df_analise)} meses sincronizados.")

# Primeira diferença do EPU_SA (devido à raiz unitária em nível)
df_analise['d_EPU_SA'] = df_analise['EPU_SA'].diff()
df_analise_clean = df_analise.dropna().copy()

# Padronização de todas as séries para Base Média = 100 na amostra
df_analise['VIX_PCA_100'] = (df_analise['VIX_PCA'] / df_analise['VIX_PCA'].mean()) * 100
df_analise['VIX_LASSO_100'] = (df_analise['VIX_LASSO'] / df_analise['VIX_LASSO'].mean()) * 100
df_analise['EPU_Base100'] = (df_analise['EPU_SA'] / df_analise['EPU_SA'].mean()) * 100

# ==============================================================================
# 2. VERIFICAÇÃO DE ESTACIONARIEDADE (TESTE ADF)
# ==============================================================================
print("\n--- TESTE DE ESTACIONARIEDADE (ADF) ---")
p_vix_pca = adfuller(df_analise['VIX_PCA'])[1]
p_vix_lasso = adfuller(df_analise['VIX_LASSO'])[1]
p_epu = adfuller(df_analise['EPU_SA'])[1]
p_depu = adfuller(df_analise_clean['d_EPU_SA'])[1]

print(f"-> P-valor VIX Tupiniquim (PCA - Nível): {p_vix_pca:.4f}")
print(f"-> P-valor VIX Tupiniquim (LASSO - Nível): {p_vix_lasso:.4f}")
print(f"-> P-valor EPU_SA (Nível): {p_epu:.4f}")
print(f"-> P-valor EPU_SA (Primeira Diferença): {p_depu:.4e} (Estacionário)")

# ==============================================================================
# 3. TESTE DE CAUSALIDADE DE GRANGER (Lags 1 a 3)
# ==============================================================================
max_lags = 3
print(f"\n--- EXECUTANDO TESTE DE CAUSALIDADE DE GRANGER (Lags 1 a {max_lags}) ---")

# --- 3.1. VIX (PCA) vs EPU ---
print("\n[ABORDAGEM 1 - PCA]:")
print("1.1. VIX Tupiniquim (PCA) -> EPU (Notícias/Mídia)")
gc_pca_to_epu = grangercausalitytests(df_analise_clean[['d_EPU_SA', 'VIX_PCA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_pca_to_epu[lag][0]['ssr_ftest'][1]
    f_stat = gc_pca_to_epu[lag][0]['ssr_ftest'][0]
    status = "REJEITA H0 (Causa!)" if p_val < 0.05 else "Não Rejeita H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-valor = {p_val:.4f} | {status}")

print("\n1.2. EPU (Notícias/Mídia) -> VIX Tupiniquim (PCA)")
gc_epu_to_pca = grangercausalitytests(df_analise_clean[['VIX_PCA', 'd_EPU_SA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_pca[lag][0]['ssr_ftest'][1]
    f_stat = gc_epu_to_pca[lag][0]['ssr_ftest'][0]
    status = "REJEITA H0 (Causa!)" if p_val < 0.05 else "Não Rejeita H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-valor = {p_val:.4f} | {status}")

# --- 3.2. VIX (LASSO) vs EPU ---
print("\n[ABORDAGEM 2 - LASSO]:")
print("2.1. VIX Tupiniquim (LASSO) -> EPU (Notícias/Mídia)")
gc_lasso_to_epu = grangercausalitytests(df_analise_clean[['d_EPU_SA', 'VIX_LASSO']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_lasso_to_epu[lag][0]['ssr_ftest'][1]
    f_stat = gc_lasso_to_epu[lag][0]['ssr_ftest'][0]
    status = "REJEITA H0 (Causa!)" if p_val < 0.05 else "Não Rejeita H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-valor = {p_val:.4f} | {status}")

print("\n2.2. EPU (Notícias/Mídia) -> VIX Tupiniquim (LASSO)")
gc_epu_to_lasso = grangercausalitytests(df_analise_clean[['VIX_LASSO', 'd_EPU_SA']], maxlag=max_lags, verbose=False)
for lag in range(1, max_lags + 1):
    p_val = gc_epu_to_lasso[lag][0]['ssr_ftest'][1]
    f_stat = gc_epu_to_lasso[lag][0]['ssr_ftest'][0]
    status = "REJEITA H0 (Causa!)" if p_val < 0.05 else "Não Rejeita H0"
    print(f"   -> Lag {lag}: F-stat = {f_stat:.4f} | P-valor = {p_val:.4f} | {status}")

# ==============================================================================
# 4. GERAÇÃO DOS GRÁFICOS SEPARADOS (EIXO DUPLO + VIA NEGATIVA: SEM LEGENDA)
# ==============================================================================

# --- 4.1. GRÁFICO 1: VIX (PCA) vs EPU (EIXO DUPLO) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

# Eixo Esquerdo (Primary Y): VIX (PCA)
color1 = 'navy'
ax1.set_xlabel('Anos', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via PCA (Base Média = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_PCA_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

# Eixo Direito (Secondary Y): EPU (Texto rotacionado)
ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Índice EPU Brasil (Base Média = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_pca_vs_epu_pt.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n[IMAGEM GERADA]: 'vix_pca_vs_epu_pt.png' (Sem legenda, eixo direito rotacionado) salva com sucesso!")


# --- 4.2. GRÁFICO 2: VIX (LASSO) vs EPU (EIXO DUPLO) ---
fig, ax1 = plt.subplots(figsize=(12, 4.5))

# Eixo Esquerdo (Primary Y): VIX (LASSO)
color1 = 'firebrick'
ax1.set_xlabel('Anos', fontsize=10, fontweight='bold')
ax1.set_ylabel('VIX Tupiniquim via LASSO (Base Média = 100)', color=color1, fontsize=10, fontweight='bold')
ax1.plot(df_analise['Data'], df_analise['VIX_LASSO_100'], color=color1, linewidth=2.0)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axhline(100, color='black', linestyle=':', linewidth=0.8, alpha=0.7)
ax1.grid(True, linestyle=':', alpha=0.6)

# Eixo Direito (Secondary Y): EPU (Texto rotacionado)
ax2 = ax1.twinx()
color2 = 'darkorange'
ax2.set_ylabel('Índice EPU Brasil (Base Média = 100)', color=color2, fontsize=10, fontweight='bold', rotation=270, labelpad=18)
ax2.plot(df_analise['Data'], df_analise['EPU_Base100'], color=color2, linewidth=1.5, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.savefig('vix_lasso_vs_epu_pt.png', dpi=300, bbox_inches='tight')
plt.show()
print("[IMAGEM GERADA]: 'vix_lasso_vs_epu_pt.png' (Sem legenda, eixo direito rotacionado) salva com sucesso!")

print("\n--- PROCESSAMENTO CONCLUÍDO COM SUCESSO! ---")